# 🛡️ NIDS Google Colab Setup & Environment Persistence
This notebook initializes the Colab environment for the **Smart-AI-Network-Intrusion-Detection-System**.

It fulfills the following requirements:
1. Mounts Google Drive to make the runtime persistent.
2. Creates the permanent `NIDS_Project` directory and subdirectories in Google Drive.
3. Installs required dependencies idempotently.
4. Automatically distinguishes between **FIRST-TIME SETUP** and **RETURNING SESSION**.
5. Provides a simple checkpoint mechanism.

In [ ]:
import os
import json
from google.colab import drive

# --- Configuration ---
PROJECT_ROOT = '/content/drive/MyDrive/NIDS_Project'
CONFIG_FILE = os.path.join(PROJECT_ROOT, 'env_config.json')

# Define required subdirectories inside Google Drive for persistence
DIRECTORIES = [
    'datasets',
    'models',
    'preprocessing',
    'results',
    'checkpoints'
]

In [ ]:
# --- Mount Google Drive ---
print("Attempting to mount Google Drive...")
drive.mount('/content/drive')
print("Google Drive mounted successfully.")

In [ ]:
# --- Environment & Project Structure Initialization ---
def setup_environment():
    if os.path.exists(CONFIG_FILE):
        print("\n\u2705 RETURNING SESSION: Environment configuration already exists.")
        with open(CONFIG_FILE, 'r') as f:
            config = json.load(f)
        print(f"Project originally initialized on: {config.get('initialized_at')}")
        print("Skipping first-time setup directory creation.")
        return config
    
    print("\n\ud83d\ude80 FIRST-TIME SETUP: Initializing project directories on Google Drive...")
    
    # Create base project directory
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    
    # Create subdirectories
    for d in DIRECTORIES:
        dir_path = os.path.join(PROJECT_ROOT, d)
        os.makedirs(dir_path, exist_ok=True)
        print(f"  Created: {dir_path}")
        
    # Save config
    from datetime import datetime
    config = {
        'initialized_at': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'project_root': PROJECT_ROOT,
        'status': 'Environment Initialized',
        'checkpoints': {
            'environment_setup': True,
            'data_uploaded_to_drive': False,
            'data_preprocessed': False,
            'model_trained': False
        }
    }
    
    with open(CONFIG_FILE, 'w') as f:
        json.dump(config, f, indent=4)
        
    print("\n\u2705 First-time setup complete! Configuration saved to Google Drive.")
    return config

config = setup_environment()

In [ ]:
# --- Checkpoint Management ---
def update_checkpoint(step_name, status=True):
    """Updates the persistent JSON checkpoint file."""
    if not os.path.exists(CONFIG_FILE):
        print("\u26a0\ufe0f Config file not found. Run setup_environment() first.")
        return
        
    with open(CONFIG_FILE, 'r') as f:
        cfg = json.load(f)
    
    if 'checkpoints' not in cfg:
        cfg['checkpoints'] = {}
        
    cfg['checkpoints'][step_name] = status
    
    with open(CONFIG_FILE, 'w') as f:
        json.dump(cfg, f, indent=4)
    print(f"\u2705 Checkpoint '{step_name}' updated to {status}.")

def check_checkpoint(step_name):
    """Checks if a particular step has already been completed."""
    if not os.path.exists(CONFIG_FILE):
        return False
    with open(CONFIG_FILE, 'r') as f:
        cfg = json.load(f)
    return cfg.get('checkpoints', {}).get(step_name, False)

print("\n\ud83d\udcca Current Project Progress:")
for k, v in config.get('checkpoints', {}).items():
    status_icon = "\u2705 Done" if v else "\u274c Pending"
    print(f" - {k}: {status_icon}")

In [ ]:
# --- Dependencies Installation ---
# Pip checks idempotently. It won't re-download if packages exist in the environment.
print("Installing/Verifying required dependencies...")
!pip install -q pandas numpy scikit-learn xgboost flask pyarrow fastparquet joblib tqdm
print("\u2705 Dependencies ready.")